# Final Gradient Descent training on Fritz

Train sparse Numba GD on the full training set with the hyperparameters selected by the fine search, then evaluate it once on the test set.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation import mae_from_predictions, rmse_from_predictions
from gd_sparce_numba import gradient_descent_numba, _observed_predictions

## Load the processed data

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml-latest"
RESULTS_DIR = PROJECT_ROOT / "results" / "gd_final"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_ratings.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_ratings.csv")
user_map = pd.read_csv(PROCESSED_DIR / "user_map.csv")
movie_map = pd.read_csv(PROCESSED_DIR / "movie_map.csv")

n_users = len(user_map)
n_movies = len(movie_map)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of users:", n_users)
print("Number of movies:", n_movies)

## Selected hyperparameters

In [ ]:
best_gd_rank = 60
best_gd_eta = 2e-4
best_gd_reg = 1.0
best_gd_init_scale = 0.01
best_gd_n_iter = 177

print("Final GD rank:", best_gd_rank)
print("Final GD eta:", best_gd_eta)
print("Final GD reg:", best_gd_reg)
print("Final GD init scale:", best_gd_init_scale)
print("Final GD iterations:", best_gd_n_iter)

## Retrain on the full training set

In [ ]:
U_gd, V_gd, gd_global_mean, gd_history = gradient_descent_numba(
    train_df,
    n_users=n_users,
    n_movies=n_movies,
    rank=best_gd_rank,
    eta=best_gd_eta,
    reg=best_gd_reg,
    n_iter=best_gd_n_iter,
    init_scale=best_gd_init_scale,
    random_state=42,
    objective_tol=None,
)

## Test-set evaluation

In [ ]:
test_user_idx = test_df["user_idx"].to_numpy(dtype=np.int64)
test_movie_idx = test_df["movie_idx"].to_numpy(dtype=np.int64)
test_ratings = test_df["rating"].to_numpy(dtype=np.float64)

gd_test_predictions = gd_global_mean + _observed_predictions(
    U_gd,
    V_gd,
    test_user_idx,
    test_movie_idx,
)
gd_test_predictions = np.clip(gd_test_predictions, 0.5, 5.0)
gd_test_rmse = rmse_from_predictions(test_ratings, gd_test_predictions)
gd_test_mae = mae_from_predictions(test_ratings, gd_test_predictions)

test_metrics_df = pd.DataFrame([{
    "rank": best_gd_rank,
    "eta": best_gd_eta,
    "reg": best_gd_reg,
    "init_scale": best_gd_init_scale,
    "n_iter": best_gd_n_iter,
    "test_rmse": gd_test_rmse,
    "test_mae": gd_test_mae,
}])
test_metrics_df.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
display(test_metrics_df)

## Training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Sparse Numba GD Training History "
    f"(rank={best_gd_rank}, eta={best_gd_eta}, reg={best_gd_reg})",
    fontsize=16,
)

axes[0].plot(gd_history["iteration"], gd_history["objective"])
axes[0].set_title("Training objective")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Objective")
axes[0].grid(alpha=0.3)

axes[1].plot(
    gd_history["iteration"],
    gd_history["train_rmse"],
    color="tab:orange",
)
axes[1].set_title("Training RMSE")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("RMSE")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()